# M11 lab: techno-economic analysis inside an optimization model

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M11 · about 45 minutes, plus 90 minutes of practice at the end.

You will rebuild the lesson's plant as one linear program, solve it with the open-source HiGHS solver through `scipy.optimize.linprog(method="highs")`, and check your results against the lesson's reference block with `assert` cells:

1. The year: the plant's wind, Prairie Town's price and its average CO₂ rate, hour by hour.
2. Thirty-six representative days, three per month.
3. Money: the capital recovery factor, the annuity factor and the costs in 2025 dollars.
4. One builder for the LP: the size, the hours, the credit's two periods, a tier's cap and three boundaries.
5. The lesson's plans: your size, the model's size, the credit in its own years and in every year.
6. One LP per tier, and the best.
7. Three boundaries.
8. Practice.

**How to run it.** Put this notebook and the five data files in one folder. In Google Colab: *File → Upload notebook*, then upload the files with the folder icon on the left, then *Runtime → Run all*. Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later. It runs in under a minute.

| File (download from the lesson page) | What it holds |
|---|---|
| `m11-tea-in-models.json` | the plant, the electrolyzer's costs, the credit as the IRS states it for 2025, the counting rules, the boundaries, the reference results, and lesson 13's wind-farm costs, rate and life (2025 dollars, its block `lesson13`) |
| `a16-hydrogen-ptx.json` | A16's electrolyzer: 57 kWh per kg |
| `town.json` | Prairie Town's fleet: sizes, running costs and CO₂ rates |
| `town_year.json` | the 8,760 hourly shapes of 2025 (Southwest Power Pool, EIA-930), in thousandths of each series' maximum |
| `cf_targets.json` | EIA's 2025 capacity factors (wind 33.8%) |

**The data.** The plant, the buyer, the charge and Prairie Town are fictional and illustrative; the shapes are real. Costs are in real 2025 dollars. The credit's amounts are the law's (IRS, Instructions for Form 7210, 2025); this lab never varies them.

In [ ]:
import json
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import linprog
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
FILES = ["m11-tea-in-models.json", "a16-hydrogen-ptx.json", "town.json", "town_year.json", "cf_targets.json"]
for name in FILES:
    assert Path(name).exists(), f"{name} not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D, A16, town, year, targets = (json.load(open(n, encoding="utf-8")) for n in FILES)
REF = D["reference"]
C13 = D["lesson13"]   # lesson 13's wind farm (installed cost, upkeep), rate and life, as lesson 13's page shows them
print("NumPy", np.__version__, "· SciPy", scipy.__version__)
print(year["meta"]["source"])

## 1. The year

**The plant's wind.** The shape $s_t$ (thousandths of its 2025 maximum) is raised to a power $g$ so the year averages EIA's 33.8%: output $= 100\,s_t^{\,g}$ MW (the lesson's method "power": the best hour stays at 100 MW, nothing is clipped). $g$ comes from bisection; values are rounded to 0.01 MW, as on the page.

**Prairie Town's price** (lesson 9): demand scaled to a 450 MW peak, its wind and solar farms scaled to their sizes; what is left after the flat plant, wind and sun is met by the combined-cycle plant ($40) and then the peaker ($100); $0 when nothing is left. **Its average CO₂ rate** is the gas plants' CO₂ divided by all the power used, over all 8,760 hours (operating CO₂ only).

In [ ]:
def js_round(x):
    return np.floor(np.asarray(x) + 0.5)   # JavaScript's Math.round: halves up

s = np.maximum(np.asarray(year["wind"], float), 0) / 1000
cf = targets["targets"]["wind"]["cf"]
cf_of = lambda g: np.where(s > 0, np.minimum(1, s) ** g, 0).mean()
lo, hi = 1e-6, 1.0
while cf_of(hi) > cf:
    lo, hi = hi, hi * 2
for _ in range(200):
    if hi - lo <= 1e-13 * max(1, hi):
        break
    mid = (lo + hi) / 2
    lo, hi = (mid, hi) if cf_of(mid) >= cf else (lo, mid)
g = (lo + hi) / 2
W = js_round(D["plant"]["windMW"] * np.where(s > 0, np.minimum(1, s) ** g, 0) * 100) / 100

f = town["fleet"]
dem = js_round(np.asarray(year["demand"]) / 1000 * D["town"]["demandPeakMW"])
flat = [v["mw"] for v in f.values() if v["behaviour"] == "runs flat"][0]   # the plant that runs flat all year
gap = dem - flat - js_round(np.asarray(year["wind"]) / 1000 * f["wind"]["mw"]) - js_round(np.asarray(year["solar"]) / 1000 * f["solar"]["mw"])
price = np.where(gap <= 0, 0, np.where(gap <= f["cc"]["mw"], f["cc"]["cost"], f["peaker"]["cost"])).astype(float)
cc = np.clip(gap, 0, f["cc"]["mw"]); pk = np.clip(gap - f["cc"]["mw"], 0, f["peaker"]["mw"])
town_rate = (cc * f["cc"]["co2"] + pk * f["peaker"]["co2"]).sum() / dem.sum()
cc_rate = f["cc"]["co2"]
print(f"wind capacity factor {W.mean() / 100:.4f} (g = {g:.4f}) · price $0 in {np.mean(price == 0):.0%} of hours · town CO2 rate {town_rate:.4f} t/MWh")
assert abs(W.mean() / 100 - cf) < 1e-4
assert round(town_rate, 3) == REF["townRate"]

## 2. Thirty-six representative days

In each month, sort the days by the plant's daily wind energy and split them into thirds (calm, middle, windy; ties to the earlier day). In each third, the **medoid** day stands for the others: the day with the least summed squared distance to them, over its 24 hours of wind per MW (divided by the windiest hour) and price per \$100. Each representative day weighs as many days as its third has: 365 in all. Days run midnight to midnight, Central Standard Time (index 24 k to 24 k + 23).

In [ ]:
MDAYS = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]
wmax = W.max()
days, k0 = [], 0
for m, nd in enumerate(MDAYS):
    ks = sorted(range(k0, k0 + nd), key=lambda k: (W[24 * k:24 * k + 24].sum(), k))
    for third in range(3):
        grp = ks[round(third * nd / 3):round((third + 1) * nd / 3)]
        feat = {k: np.concatenate([W[24 * k:24 * k + 24] / wmax, price[24 * k:24 * k + 24] / 100]) for k in grp}
        cost = {k: sum(((feat[k] - feat[j]) ** 2).sum() for j in grp) for k in grp}
        best = min(sorted(grp), key=lambda k: cost[k])
        days.append(dict(m=m, third=third, day=best, weight=len(grp), w=W[24 * best:24 * best + 24], p=price[24 * best:24 * best + 24]))
    k0 += nd
dates = [str(np.datetime64("2025-01-01") + np.timedelta64(d["day"], "D")) for d in days]
print(len(days), "days, weights add to", sum(d["weight"] for d in days))
print("representative days' wind capacity factor:", round(sum(d["weight"] * d["w"].sum() for d in days) / 876000, 4), "(the year: 0.338)")
assert dates == [d["date"] for d in REF["days"]], "the days differ from the lesson's"

## 3. Money in real 2025 dollars

$\mathit{CRF}(r, N) = r / (1 - (1+r)^{-N})$ spreads an up-front cost evenly over $N$ years; the annuity factor $\mathit{AF}(r, N) = 1 / \mathit{CRF}$ is what \$1 a year for $N$ years is worth today. DOE's \$2,000 per kW (2022 dollars) becomes 2025 dollars with BEA's GDP deflator; upkeep is 5% of it a year and stack wear 11% of it per 40,000 full-load hours, charged per MWh.

In [ ]:
crf = lambda r, n: 1 / n if r == 0 else r / (1 - (1 + r) ** -n)
af = lambda r, n: n if r == 0 else (1 - (1 + r) ** -n) / r
idx = D["deflator"]["index"]
def costs(cap="mid", r=C13["finance"]["wacc"] / 100, N=C13["finance"]["life"]):
    capex = D["electrolyzer"]["capex2022"][cap] * idx["2025"] / idx["2022"]
    E, Wd = D["electrolyzer"], C13["techs"]["wind"]
    return dict(capexKW=capex, r=r, N=N, crf=crf(r, N), af=af(r, N), fomKW=E["fomShare"] * capex, wear=E["replaceShare"] * capex * 1000 / E["replaceHours"],
                k=1000 / A16["electrolyzer"]["kWhPerKg"], windCapexKW=Wd["capex"], windFomKW=Wd["fom"], windMW=D["plant"]["windMW"], C=D["plant"]["connectionMW"])
K0 = costs()
print(f"CRF {K0['crf']:.4f} · AF {K0['af']:.2f} · capex ${K0['capexKW']:,.0f}/kW · wear ${K0['wear']:.2f}/MWh · {K0['k']:.2f} kg/MWh")
assert abs(K0["crf"] - REF["defaults"]["crf"]) < 1e-12 and abs(K0["capexKW"] - REF["defaults"]["capexKW"]) < 1e-9

## 4. One builder for the LP

**Columns** (per period $p$, day $d$, hour $t$): $\mathit{imp}$ (bought), and behind the plant's meter $\mathit{dw}$ (wind to the electrolyzer) and $\mathit{exp}$ (sold); with hourly matching across the grid $u$ (bought, not matched in the hour); and the size $S$ unless it is given. Annual matching needs no extra column: $e \max(0, \sum(\mathit{imp} - \mathit{exp})) \le L\,\mathit{kg}$ is the same as $e \sum(\mathit{imp} - \mathit{exp}) \le L\,\mathit{kg}$, since the right side is never negative. $\mathit{ez} = \mathit{dw} + \mathit{imp}$ is shorthand.

**Goal** (\$ a year, annual equivalent): $(\mathit{CRF} + f)\,c\,S + \sum_p \varphi_p \sum_{d,t} n_d [(p_t + q)\,\mathit{imp} - p_t\,\mathit{exp} + (v - (h + \kappa_p)k)\,\mathit{ez}]$, plus the tie rule's \$0.001 per MWh on power bought and against wind curtailed (left out of every result).

**Periods.** With a credit in its own years: $\varphi_1 = \mathit{AF}(r, 10)/\mathit{AF}(r, N)$ for the credit years, $\varphi_2 = 1 - \varphi_1$ after; one period otherwise.

**Bounds.** Every column is $\ge 0$ and $0 \le S \le 100$ MW; $\mathit{dw}$ is at most $w_t$. Behind the plant's meter, $\mathit{imp}$ and $\mathit{exp}$ are at most 50 MW: they share the wind farm's connection. Across the grid the electrolyzer has its own connection at the buyer, so $\mathit{imp}$ is all its power, up to $S$ (the row $\mathit{ez} - S \le 0$; with the size given, the bound $S$). The bound $u \ge 0$ matters: a negative $u$ in an hour that sells would cancel power bought in another hour, which is annual matching, not hourly. `linprog`'s `bounds` hold them.

**Rows.** $\mathit{dw} + \mathit{exp} \le w_t$; $\mathit{ez} - S \le 0$; a tier's cap in period 1: $\sum n_d (e \cdot \text{counted} - L\,k\,\mathit{ez}/1000) \le 0$, with $L$ a hair (0.001) under each "less than" limit. Behind one meter the plan never buys and sells in the same hour (the tie rule), so hourly matching counts every MWh bought and needs no $u$; across the grid $u \ge \mathit{imp} - \min(w_t, 50)$. Across the grid ($\mathit{boundary} = $ `"grid"`) there is no $\mathit{dw}$ or $\mathit{exp}$: the wind farm sells $\min(w_t, 50)$ on its own, and those sales are the certificates that matching may count.

In [ ]:
TIE = D["tie"]
def periods(K, o):
    if not o.get("credit"):
        return [dict(id=1, phi=1.0, credit=0.0, cap=False)]
    if o.get("window") == "all" or K["N"] <= 10:
        return [dict(id=1, phi=1.0, credit=o["credit"], cap=o.get("tier") is not None)]
    p1 = af(K["r"], 10) / K["af"]
    return [dict(id=1, phi=p1, credit=o["credit"], cap=o.get("tier") is not None), dict(id=2, phi=1 - p1, credit=0.0, cap=False)]

def build(o, K=K0, days=days):
    grid, rule, fixed = o["boundary"] == "grid", o.get("rule", "none"), o.get("size") is not None
    maxMW = D["plant"]["maxMW"]
    lim = (o["tier"]["max"] - (D["credit"]["strictMargin"] if o["tier"]["strict"] else 0)) if o.get("tier") else 0
    names, cost, lb, ub = [], [], [], []
    def var(n, c=0.0, lo=0.0, hi=np.inf):
        names.append(n); cost.append(c); lb.append(lo); ub.append(hi); return len(names) - 1
    rows = []                                       # (dict col -> coef, lower, upper)
    S = None if fixed else var("S", (K["crf"] + D["electrolyzer"]["fomShare"]) * K["capexKW"] * 1000, 0, maxMW)
    for pp in periods(K, o):
        cap, nb = {}, 0.0
        for di, dy in enumerate(days):
            for t in range(24):
                wt, p, w = pp["phi"] * dy["weight"], dy["p"][t], dy["w"][t]
                ezc = wt * (K["wear"] - (o["h"] + pp["credit"]) * K["k"])
                imp = var(f"imp{pp['id']}_{di}_{t}", wt * (p + o["c"] + TIE["import"]) + ezc, 0, (o["size"] if fixed else maxMW) if grid else K["C"])
                ez = [imp]
                if not grid:
                    dw = var(f"dw{pp['id']}_{di}_{t}", ezc, 0, w)
                    ex = var(f"exp{pp['id']}_{di}_{t}", -wt * (p + TIE["curtail"]), 0, min(K["C"], w))
                    rows.append(({dw: 1, ex: 1}, -np.inf, w)); ez.append(dw)
                if not fixed:
                    r = {c: 1 for c in ez}; r[S] = -1; rows.append((r, -np.inf, 0))
                elif not grid:
                    rows.append(({c: 1 for c in ez}, -np.inf, o["size"]))
                sold = min(K["C"], w) if grid else 0.0
                if pp["cap"]:
                    # behind one meter the tie rule never buys and sells in one hour, so hourly matching counts every
                    # MWh bought, as "none" does; only across the grid does it need u
                    if rule == "hourly" and grid:
                        u = var(f"u{pp['id']}_{di}_{t}", 0, 0, maxMW if grid else K["C"])
                        r = {u: 1, imp: -1}
                        if not grid:
                            r[ex] = 1
                        rows.append((r, -sold, np.inf)); cap[u] = cap.get(u, 0) + dy["weight"] * o["e"]
                    elif rule in ("none", "hourly"):
                        cap[imp] = cap.get(imp, 0) + dy["weight"] * o["e"]
                    else:   # annual: e (bought - certificates) <= limit x kg, one row (the right side is never negative)
                        cap[imp] = cap.get(imp, 0) + dy["weight"] * o["e"]
                        if grid:
                            nb += dy["weight"] * o["e"] * sold
                        else:
                            cap[ex] = cap.get(ex, 0) - dy["weight"] * o["e"]
                    for c in ez:
                        cap[c] = cap.get(c, 0) - dy["weight"] * lim * K["k"] / 1000
        if pp["cap"]:
            rows.append((cap, -np.inf, nb))
    return names, np.array(cost), list(zip(lb, ub)), rows

def solve(o, K=K0, days=days):
    names, c, bounds, rows = build(o, K, days)
    ri, ci, v, bl, bu = [], [], [], [], []
    for i, (r, lo, hi) in enumerate(rows):
        for j, a in r.items():
            ri.append(i); ci.append(j); v.append(a)
        bl.append(lo); bu.append(hi)
    A = coo_matrix((v, (ri, ci)), shape=(len(rows), len(names))).tocsr()
    bl, bu = np.array(bl), np.array(bu)
    up, low = np.isfinite(bu), np.isfinite(bl)            # every row here has one finite side
    A_ub = scipy.sparse.vstack([A[up], -A[low]]).tocsr(); b_ub = np.concatenate([bu[up], -bl[low]])
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method="highs")
    assert res.status == 0, res.message
    return read(o, K, days, dict(zip(names, res.x))), res

**Reading a plan.** Each period's year (weighted by the days): kilograms, power bought and its cost, wind sold, stack wear, hydrogen revenue and credit. Then the cash flows: year 0 the build (the wind farm too for the one-plant boundary), years 1 to 10 the credit years, the rest after; NPV, the annual equivalent (NPV ÷ AF), and the **cost per kg before credits**: discounted costs net of power sales ÷ discounted kilograms. **Capacity utilization** is the credit years' MWh into the electrolyzer ÷ ($S$ × 8,760 h), its average loading: running at half power all year gives 50%, the same as full power for half the year. For the **increment**, the wind farm on its own (selling what its 50 MW connection takes) is subtracted: same plan, different accounting.

In [ ]:
def read(o, K, days, x):
    v = lambda n: x.get(n, 0.0)
    b, grid, S = o["boundary"], o["boundary"] == "grid", (o["size"] if o.get("size") is not None else x["S"])
    per = []
    for pp in periods(K, o):
        a = dict(ez=0, imp=0, exp=0, bought=0, sales=0, alone=0, hourly=0, certs=0, credit=pp["credit"])
        for di, dy in enumerate(days):
            for t in range(24):
                n, wt = f"{pp['id']}_{di}_{t}", dy["weight"]
                im = v("imp" + n); a["imp"] += wt * im; a["ez"] += wt * (im + (0 if grid else v("dw" + n)))
                a["bought"] += wt * (dy["p"][t] + o["c"]) * im; a["alone"] += wt * dy["p"][t] * min(K["C"], dy["w"][t])
                sold = min(K["C"], dy["w"][t]) if grid else v("exp" + n)
                if not grid:
                    a["exp"] += wt * v("exp" + n); a["sales"] += wt * dy["p"][t] * v("exp" + n)
                a["hourly"] += wt * max(0, im - sold); a["certs"] += wt * sold
        a["kg"] = a["ez"] * K["k"]
        rule = o.get("rule", "none")
        a["counted"] = a["hourly"] if rule == "hourly" else max(0, a["imp"] - a["certs"]) if rule == "annual" else a["imp"]
        a["rate"] = a["counted"] * o.get("e", 0) * 1000 / a["kg"] if a["kg"] > 0 else 0
        sales = a["sales"] if b == "plant" else a["sales"] - a["alone"] if b == "increment" else 0
        a["costs"] = a["bought"] + K["wear"] * a["ez"] + K["fomKW"] * 1000 * S + (K["windFomKW"] * 1000 * K["windMW"] if b == "plant" else 0) - sales
        a["net"] = o["h"] * a["kg"] + a["credit"] * a["kg"] - a["costs"]
        per.append(a)
    capex0 = K["capexKW"] * 1000 * S + (K["windCapexKW"] * 1000 * K["windMW"] if b == "plant" else 0)
    yrs = min(K["N"], 10) if (o.get("credit") and o.get("window") != "all") else K["N"]
    flows, kgPV, costPV, credPV = [-capex0], 0.0, capex0, 0.0
    for y in range(1, K["N"] + 1):
        a = per[1] if len(per) > 1 and y > yrs else per[0]; df = (1 + K["r"]) ** -y
        flows.append(a["net"]); kgPV += a["kg"] * df; costPV += a["costs"] * df; credPV += a["credit"] * a["kg"] * df
    npv = sum(fl * (1 + K["r"]) ** -y for y, fl in enumerate(flows))
    return dict(S=S, per=per, flows=flows, npv=npv, annual=npv / K["af"], kg=per[0]["kg"], rate=per[0]["rate"],
                lcoh=costPV / kgPV if kgPV > 0 else None, creditLev=credPV / kgPV if kgPV > 0 else 0.0,
                util=per[0]["ez"] / (S * 8760) if S > 1e-6 else 0.0, x=x)

def show(lab, R):
    lc = "–" if R["lcoh"] is None else f"${R['lcoh']:.2f}/kg"
    print(f"{lab:34s} {R['S']:5.1f} MW · {R['kg'] / 1000:6,.0f} t a year · utilization {R['util']:.0%} · NPV ${R['npv'] / 1e6:7.1f}M · {lc} · counted grid CO2 (operating) {R['rate']:.3f} kg/kg")

## 5. The lesson's plans

The defaults: hydrogen at \$4 per kg, a \$10 per MWh delivery charge, Prairie Town's grid rate, hourly matching, DOE's middle cost, 4% real over 30 years, one plant (wind farm and electrolyzer). The top tier pays $5 \times \$0.637 = \$3.185$ per kg (wage and apprenticeship rules met). Each `assert` compares with the lesson's reference block: sizes to 0.1 MW, hydrogen a year to 0.5%, NPVs to \$0.1M, the cost per kg before credits to \$0.01, and the credit per kg over the life to \$0.005. The tolerances allow for JavaScript and NumPy rounding a few hours differently, and for plans of the same value that run some hours differently.

In [ ]:
T1 = D["credit"]["tiers"][0]
TOP = T1["amount"] * D["credit"]["multiplier"]
base = dict(boundary="plant", h=D["hydrogen"]["price"]["def"], c=D["charge"]["def"], rule="hourly", e=town_rate)
plans = dict(fixed=dict(base, size=D["fixedSize"]["def"]), sized=dict(base),
             t1=dict(base, credit=TOP, tier=T1, window="right"), t1all=dict(base, credit=TOP, tier=T1, window="all"),
             annual=dict(base, rule="annual", credit=TOP, tier=T1, window="right"))
R = {}
for k, o in plans.items():
    R[k], _ = solve(o)
    show(k, R[k])
    ref = REF["defaults"][k]
    assert abs(R[k]["S"] - ref["S"]) < 0.1, (k, "S", R[k]["S"], ref["S"])
    assert abs(R[k]["kg"] - ref["kg"]) < 0.005 * ref["kg"], (k, "kg", R[k]["kg"], ref["kg"])
    assert abs(R[k]["npv"] - ref["npv"]) < 1e5, (k, "npv", R[k]["npv"], ref["npv"])
    assert (R[k]["lcoh"] is None) == (ref["lcoh"] is None) and (ref["lcoh"] is None or abs(R[k]["lcoh"] - ref["lcoh"]) < 0.01), (k, "lcoh", R[k]["lcoh"], ref["lcoh"])
    assert abs(R[k]["creditLev"] - ref["creditLev"]) < 0.005, (k, "creditLev", R[k]["creditLev"], ref["creditLev"])
x = R["t1"]; share = x["creditLev"] / TOP   # the credit years' share of the discounted kilograms
print(f"\nthe credit in its own years: worth ${x['creditLev']:.2f}/kg over the life ({TOP} x {share:.3f});")
print(f"  {x['per'][0]['kg'] / 1e6:.2f} million kg a year under the cap, {x['per'][1]['kg'] / 1e6:.2f} million after it")
print(f"in every year the NPV is ${(R['t1all']['npv'] - R['t1']['npv']) / 1e6:.1f}M higher")
assert abs(x["per"][1]["kg"] - REF["defaults"]["t1"]["kgAfter"]) < 0.005 * REF["defaults"]["t1"]["kgAfter"] and abs(share - REF["defaults"]["t1"]["kgShare"]) < 0.002

**Check the identity.** When every year of a period is alike, $\mathrm{NPV} = \mathit{AF}(r, N) \times$ the annual equivalent, and the year-by-year flows discount to the same number. The credit is paid in years 1 to 10 only.

In [ ]:
x = R["t1"]
assert abs(sum(f * (1 + K0["r"]) ** -y for y, f in enumerate(x["flows"])) - x["annual"] * K0["af"]) < 1
assert x["per"][1]["credit"] == 0 and abs((x["flows"][10] - x["flows"][11]) - (x["per"][0]["net"] - x["per"][1]["net"])) < 1e-6
for y in (0, 1, 10, 11, 30):
    print(f"year {y:2d}: ${x['flows'][y] / 1e6:8.2f}M")

**One day's hours.** The top tier's plan on one representative day (March, its windy third), in its credit years: MW over each hour. Printed, not asserted: other plans of the same value can run some hours differently.

In [ ]:
di = 8   # the page's default day: March, windy third
dy, x = days[di], R["t1"]["x"]
print(f"{dates[di]} (stands for {dy['weight']} days)")
print("hour   price $/MWh   wind   electrolyzer   bought   sold")
for t in range(24):
    n = f"1_{di}_{t}"
    im, dw, ex = x.get("imp" + n, 0.0), x.get("dw" + n, 0.0), x.get("exp" + n, 0.0)
    print(f"{t:02d}:00 {dy['p'][t]:9.0f} {dy['w'][t]:10.1f} {im + dw:10.1f} {im:12.1f} {ex:6.1f}")

## 6. One LP per tier

The credit is a step in the emissions rate: no single LP can hold the jump. Solve one LP with no credit and one per tier (its amount and its cap). The largest annual equivalent wins; of the cases within \$1 a year of it, the one listed first (no credit, then the cleaner tier).

**A case is a candidate, not a classification.** Each case caps the counted CO₂ from above only, so its plan can be cleaner than its tier and earn less than it could (at the defaults the 4.0 case's plan counts about 1.85 kg per kg). Such a case never wins: the cleaner tier's LP holds the same plan and pays more. So the best case's plan lies in its own tier, which the last `assert` below checks.

**The page's bound.** Without its cap, a case's hours are independent once the size $S$ is set, so its least net cost is a convex, piecewise-linear function of $S$, minimized by sweeping its corners: no LP. Minus that least cost, without the tie rule's hair costs, is an upper bound on the value the capped case can report. The page orders the tiers by their bounds, best first, with no credit last, and skips a case that cannot win even at its bound: (a) its bound is more than \$2 a year below the best LP solved so far (\$1 for the tie rule, \$1 for the solver's rounding), so it cannot come within \$1 of the best; or (b) a case listed before it is solved and reports at least its bound, so that earlier case would win any tie. The cells below solve all five, check that every bound is a bound, and count the LPs the page solves.

In [ ]:
TIE_USD, GUARD = 1, 1
def cases_of(wage=True):
    return [("none", None, None)] + [(t["id"], t, t["amount"] * (D["credit"]["multiplier"] if wage else 1)) for t in D["credit"]["tiers"]]

def pick(out):
    # the largest annual equivalent; of the cases within $1 a year of it, the one listed first; no hydrogen claims nothing
    top = max(Rc["annual"] for _, Rc in out)
    best = next((cid, Rc) for cid, Rc in out if Rc["annual"] >= top - TIE_USD)
    return out[0] if best[1]["kg"] < 1 else best

def evaluate(o, K=K0, wage=True):
    out = [(cid, solve(dict(o, credit=cr, tier=t, window="right"), K)[0]) for cid, t, cr in cases_of(wage)]
    return out, pick(out)

cases, best = evaluate(base)
CAPS = {t["id"]: ("under " if t["strict"] else "at most ") + str(t["max"]) for t in D["credit"]["tiers"]}
for cid, Rc in cases:
    show("no credit" if cid == "none" else f"{cid}: its credit, CO2 {CAPS[cid]}", Rc)
print("best:", best[0])
# The page's bound: without its cap, a case's hours are independent once the size S is set, so its least net cost is a
# convex, piecewise-linear function of S (each hour: wind the connection cannot carry, then wind it could sell, then power
# bought). Its negative bounds the case's value: a case whose bound cannot beat the best LP found so far needs no LP;
# "none" is exact.
def free_obj(o, K=K0, days=days, hair=True):
    grid, maxMW, ev, base = o["boundary"] == "grid", D["plant"]["maxMW"], [], 0.0
    tie = TIE if hair else dict(TIE, **{"import": 0.0, "curtail": 0.0})
    cap_cost = (K["crf"] + D["electrolyzer"]["fomShare"]) * K["capexKW"] * 1000
    def seg(start, ln, m):
        if m > 0 and ln > 0 and start < maxMW:
            ev.append((start, -m)); ev.append((min(maxMW, start + ln), m))
    for pp in periods(K, o):
        for dy in days:
            for t in range(24):
                wt, p, w = pp["phi"] * dy["weight"], dy["p"][t], dy["w"][t]
                V = (o["h"] + pp["credit"]) * K["k"] - K["wear"]
                if grid:
                    seg(0, maxMW, wt * (V - p - o["c"] - tie["import"])); continue
                spill, sell = max(0, w - K["C"]), min(w, K["C"])
                base -= wt * (p + tie["curtail"]) * sell
                seg(0, spill, wt * V); seg(spill, sell, wt * (V - p - tie["curtail"])); seg(spill + sell, K["C"], wt * (V - p - o["c"] - tie["import"]))
    ev.sort(key=lambda e: e[0])
    pos, F, slope, best = 0.0, base, cap_cost, base
    i = 0
    while i < len(ev):
        x = ev[i][0]; F += slope * (x - pos); pos = x; best = min(best, F)
        while i < len(ev) and ev[i][0] == x:
            slope += ev[i][1]; i += 1
    return min(best, F + slope * (maxMW - pos))

def annual_bound(o, K=K0, days=days):
    # the uncapped optimum with no hair costs, as an annual equivalent the way read() reports it
    b = o["boundary"]
    wind_c = K["windFomKW"] * 1000 * K["windMW"] + K["crf"] * K["windCapexKW"] * 1000 * K["windMW"]
    alone = sum(dy["weight"] * dy["p"][t] * min(K["C"], dy["w"][t]) for dy in days for t in range(24))
    return -free_obj(o, K, days, hair=False) - (wind_c if b == "plant" else 0) - (alone if b == "increment" else 0)

objs = {cid: solve(dict(base, credit=cr, tier=t, window="right"))[1].fun for cid, t, cr in cases_of()}
frees = {cid: free_obj(dict(base, credit=cr, tier=None, window="right")) for cid, t, cr in cases_of()}
assert all(frees[c] <= objs[c] + 1e-3 for c in objs) and abs(frees["none"] - objs["none"]) < 1e-3   # the LP's bound; exact for "none"
ubs = {cid: annual_bound(dict(base, credit=cr, tier=None, window="right")) for cid, t, cr in cases_of()}
annual = dict(cases)
assert all(annual[c]["annual"] <= ubs[c] + 0.5 for c in ubs), "a case reported more than its bound"
top, solved, done, order = -np.inf, 0, set(), [c for c, _, _ in cases_of()]   # order: pick's listing
for cid in sorted([c for c in ubs if c != "none"], key=lambda c: -ubs[c]) + ["none"]:
    earlier = [c for c in order[:order.index(cid)] if c in done and annual[c]["annual"] >= ubs[cid] - 0.01]
    if ubs[cid] < top - TIE_USD - GUARD or earlier:
        why = f"it cannot come within $1 of the best LP so far (${top / 1e6:.2f}M)" if ubs[cid] < top - TIE_USD - GUARD else f"an earlier case ({earlier[0]}) already reports as much"
        print(f"{cid:5s} skipped: its bound is ${ubs[cid] / 1e6:.2f}M a year and {why}")
        continue
    top, solved = max(top, annual[cid]["annual"]), solved + 1
    done.add(cid)
print("LPs the page solves at the defaults:", solved, "of 5")
assert solved == REF["defaults"]["solved"]
assert best[0] == REF["defaults"]["best"]
# the best case's plan lies in its own tier: at or above the cleaner tier's cap (a cleaner plan would win that tier)
tiers = D["credit"]["tiers"]; i = [t["id"] for t in tiers].index(best[0]) if best[0] != "none" else -1
if i > 0:
    lo = tiers[i - 1]["max"] - (D["credit"]["strictMargin"] if tiers[i - 1]["strict"] else 0)
    assert best[1]["rate"] >= lo - 1e-6, "the best plan is cleaner than its tier"
for (cid, Rc), ref in zip(cases, REF["defaults"]["tiers"]):
    assert cid == ref["id"] and abs(Rc["S"] - ref["S"]) < 0.1 and abs(Rc["annual"] - ref["annual"]) < 1e4

## 7. Three boundaries

The same wind and the same buyer, three boundaries: **one plant** (wind farm and electrolyzer, with the wind farm's build and upkeep and its power sales); **the electrolyzer's increment** (the same plan, minus the wind farm on its own); **across the grid** (the electrolyzer at the buyer takes every MWh from the grid and pays the charge on all of it; the wind farm's own sales are its certificates). Each with the best tier, and with no credit.

In [ ]:
for b, ref in zip(["plant", "increment", "grid"], REF["boundaries"]):
    _, (cid, Rb) = evaluate(dict(base, boundary=b))
    Rn, _ = solve(dict(base, boundary=b))
    show(f"{b} ({cid})", Rb); show(f"{b}, no credit", Rn)
    assert cid == ref["tier"] and abs(Rb["S"] - ref["S"]) < 0.1 and abs(Rb["npv"] - ref["npv"]) < 1e5 and abs(Rn["npv"] - ref["npvNone"]) < 1e5

## 8. Practice (about 90 minutes)

1. **Predict, then solve.** At \$5 per kg, will the top tier still win for one plant? Will the model build more or less? Run `evaluate(dict(base, h=5))`, then explain the size in one sentence that names the binding cap.
2. **A dirtier grid.** Count bought power at the combined-cycle plant's rate (`e=cc_rate`, 0.37 t per MWh). How much grid power does the top tier now allow per kg? Which tier wins, and why?
3. **Annual against hourly.** For the plant across the grid, compare `rule="annual"` with `rule="hourly"`. Where does the difference come from? Find it in the `read` results (`per[0]["certs"]`, `per[0]["counted"]`).
4. **Write the transition.** Assume construction begins in 2027 and the plant is placed in service on 1 January 2028 (an assumption). Its credit years 1 and 2 (2028 and 2029) use annual matching, years 3 to 10 (2030 to 2037) hourly matching, and years 11 to 30 earn no credit: `PERIODS_2028` below holds that structure (start year, end year, rule, credit). Three functions change, not one:
   - `periods()` returns one entry per row, with its weight $(\mathit{AF}(r, \text{end}) - \mathit{AF}(r, \text{start} - 1))/\mathit{AF}(r, N)$, its credit, its rule and its own cap (each credit period's emissions are capped on their own);
   - `build()` reads each period's `rule` in place of `o["rule"]`;
   - `read()` counts each period's grid power by that period's rule, and its cash-flow loop takes year $y$'s period from `start` and `end` in place of assuming two periods.

   Then compare the NPV with the two one-rule runs (`rule="annual"` and `rule="hourly"`).
5. **Check on the full year.** Fix the top tier's size and re-run its hours on all 365 days (`days365` below; a few seconds). How far is the NPV from the 36 days'? The lesson's test does this for seven plans.

In [ ]:
# Task 4: one entry per period, years counted from the day the plant is placed in service (1 January 2028, an
# assumption): annual matching for electricity until the end of 2029, hourly matching from 1 January 2030
PERIODS_2028 = [dict(id=1, start=1, end=2, rule="annual", credit=True),
                dict(id=2, start=3, end=10, rule="hourly", credit=True),
                dict(id=3, start=11, end=30, rule=None, credit=False)]
for q in PERIODS_2028:
    print(q["id"], f"years {q['start']}-{q['end']}", q["rule"], f"weight {(af(K0['r'], q['end']) - af(K0['r'], q['start'] - 1)) / K0['af']:.3f}")

days365 = [dict(m=-1, third=-1, day=k, weight=1, w=W[24 * k:24 * k + 24], p=price[24 * k:24 * k + 24]) for k in range(365)]
# Task 5:
# Rf, _ = solve(dict(plans["t1"], size=R["t1"]["S"]), days=days365)
# show("t1 on all 8,760 hours", Rf)